# Translation test: one sample paragraph

**What this notebook does:** it sends one German news paragraph to the university AI and gets the English translation back, using the professor's translation prompt.

---

## 🟢 The easy way: run everything at once

1. Check the **top-right corner**: it should say **`.venv`** or **`Python 3 (CAC .venv)`**.
   - If it says *"Select Kernel"*, click it → **Python Environments…** → choose **`.venv`**.
2. Click **▶▶ Run All** in the toolbar **at the top of this notebook**.
3. A box appears **at the top centre of VS Code**: *"Paste the Uni API key"*.
   - Open **`Howto.rmd`** (in Downloads), copy the key after `api_key =` (starts with **`sk-`**, without the quotes).
   - Paste it into the box and press **Enter**.
4. Wait about 5 seconds, then scroll down to **STEP 5** to see the translation.

**That's all.** The steps below explain what each cell does, in case you want to run them one by one.

---

⚠️ **Rules:**
- Always run the cells **from top to bottom** (STEP 1 → STEP 7). Each step needs the steps before it.
- If you **restart VS Code** or the kernel, start again from STEP 1 (or click Run All).

---
## STEP 1: Settings (connect to the university AI)

**👉 What to do:**
1. Click ▶ to the left of the cell below.
2. A box appears at the **top centre of VS Code**: *"Paste the Uni API key"*.
3. Paste the key (from `Howto.rmd`, starts with `sk-`) and press **Enter**.

**✅ You should see:** `Key loaded: True`

**What it does:** loads the tools, sets the AI address (`maki.uni-mannheim.de`) and model (`gemma4-26b`), and asks for the API key. The key is **not saved** in this file, so the file is safe to share.

In [ ]:
import os, json, re, time, getpass
import requests
import pandas as pd

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "gemma4-26b"      # the model named in the professor's Howto
TEMPERATURE = 0           # 0 = as deterministic as possible
SEED = 123

# API key: never write it into this file (we share it with the group).
# Either set the environment variable UNI_API_KEY, or paste it when asked.
API_KEY = os.environ.get("UNI_API_KEY") or getpass.getpass("Paste the Uni API key: ")
print("Key loaded:", bool(API_KEY))

---
## STEP 2: The professor's translation prompt

**👉 What to do:** click ▶ on the cell below. **Don't change anything.**

**✅ You should see:** the prompt text printed below the cell.

**What it does:** stores the professor's prompt (from ILIAS). It tells the AI to translate into English, keep names, numbers and `[Gruppe 1]`, and answer in JSON: `{"label": "<translation>"}`.

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("⚠️ Please run STEP 1 first (or click Run All at the top).")

PROFESSOR_PROMPT = """You are a careful and attentive translator for news articles.

You will receive a multiparagraph in this form:

Text: <multi-paragraph>

You are given a multi-paragraph from a news article and translate it into clear English.
Preserve names and numbers exactly, translate quotes as directly as possible.
Return only the translation.
Do not comment on the translation.

If paragraphs contain sentence fragments, assume it is a headline or image description. Translate them as sentence fragments as well. Maintain the paragraph structure.

Some entity names will be masked as [Gruppe 1] and [andere Gruppe]. Maintain those names.

Output Format:

Return only a valid JSON object:

{
  "label": "<translated text>"
}

No additional explanations, no Markdown, and no text outside the JSON."""

instructions = PROFESSOR_PROMPT
reminder = None   # the professor gave no reminder for translation
JSON_MODE = True  # the prompt asks for JSON output
print(instructions)

---
## STEP 3: The German paragraph

**👉 What to do:** click ▶ on the cell below.

**✅ You should see:** the German paragraph printed below the cell.

**💡 To translate a different paragraph:** replace the German text between the three quotes `""" … """` with your own paragraph, then run STEP 3, 5, 6 and 7 again.

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("⚠️ Please run STEP 1 first (or click Run All at the top).")

sample = pd.DataFrame({
    "id": [1],
    "text": ["""Duell zweier  Eigengewächse

Glücklich war der Nackenheimer Trainer auch über die Leistung von Akhmed Aibuev, der sich in der Klasse bis 98 Kilo (Freistil) 16:6 gegen den in dieser Saison bis dato ungeschlagenen [Gruppe 1] Akhmed Magamaev durchsetzte. "Richtig stark gerungen", attestiert Ferdinand. Allerdings hatten die SVA-Sportler auch in einigen Duellen, in denen Athleten auf Augenhöhe aufeinandertrafen, das Nachsehen. Etwa beim Kampf der Eigengewächse (75 Kg, Freistil), in dem Nackenheims Koray Cakici gegen Mansur Dakiev unbedingt seinen ersten Saisonsieg einfahren wollte, sich aber zu oft von seinem Kontrahenten abkochen ließ und 2:10 verlor. Auch sein Bruder Kubilay Cakici blieb erfolglos und verlor gegen Burghausens Ali-Pasha Umarpashaev (80 Kilo, Freistil) 2:8 - wenn auch unter kuriosen Umständen. Denn beim Zwischenstand von 2:4 pfiff der Ringrichter versehentlich, was die Sportler sichtlich irritierte. Bevor sich Cakici versah, wurde er vom schneller reagierenden [Gruppe 1] gedreht. Die Wertung zählte. "Das war nicht richtig", ärgerte sich Robin Ferdinand."""]
})
print(sample.loc[0, "text"])

---
## STEP 4: The translation function

**👉 What to do:** click ▶ on the cell below. **Don't change anything.**

**✅ You should see:** nothing. That's normal: this cell only *prepares* the function.

**What it does:** prepares the message to the AI exactly like the professor's R package `bacchuss` does (system message = prompt, user message = `Text: '<paragraph>'`).

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("⚠️ Please run STEP 1 first (or click Run All at the top).")

def call_llm(text, instructions, reminder=None, json_mode=False, max_retries=3):
    user_msg = f"Text: '{text}'"
    if reminder:
        user_msg += f"\nReminder: {reminder}"
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user",   "content": user_msg},
        ],
        "temperature": TEMPERATURE,
        "seed": SEED,
        "reasoning_effort": "none",   # same as bacchuss default (reasoning = FALSE)
    }
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(f"{HOST}/chat/completions",
                              headers={"Authorization": f"Bearer {API_KEY}"},
                              json=body, timeout=120)
            r.raise_for_status()
            j = r.json()
            return {
                "output": j["choices"][0]["message"]["content"].strip(),
                "tokens_prompt": j["usage"]["prompt_tokens"],
                "tokens_response": j["usage"]["completion_tokens"],
            }
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
            time.sleep(2 * attempt)
    return {"output": None, "tokens_prompt": None, "tokens_response": None}

---
## STEP 5: Translate ⭐

**👉 What to do:** click ▶ on the cell below and wait 1–5 seconds.

**✅ You should see:** the **English translation**, and at the end a line like:
`--- JSON parsed: True | 0.9 s | prompt tokens: 503 | response tokens: 278`

**What it does:** sends the paragraph to the AI, receives the answer and takes the translation out of the JSON `label` field.

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("⚠️ Please run STEP 1 first (or click Run All at the top).")

start = time.time()
res = call_llm(sample.loc[0, "text"], instructions, reminder, json_mode=JSON_MODE)
seconds = round(time.time() - start, 1)

raw = res["output"]
try:
    translation = json.loads(raw)["label"]      # the prompt puts the translation in "label"
    parsed_ok = True
except Exception:
    translation, parsed_ok = raw, False         # keep raw output so we can inspect it

sample.loc[0, "raw_output"] = raw
sample.loc[0, "translation"] = translation
sample.loc[0, "json_ok"] = parsed_ok
sample.loc[0, "tokens_prompt"] = res["tokens_prompt"]
sample.loc[0, "tokens_response"] = res["tokens_response"]
sample.loc[0, "seconds"] = seconds

print(translation)
print(f"\n--- JSON parsed: {parsed_ok} | {seconds} s | prompt tokens: {res['tokens_prompt']} | response tokens: {res['tokens_response']}")

---
## STEP 6: Quality checks

**👉 What to do:** click ▶ on the cell below.

**✅ You should see:** **OK** in every line:
- `JSON valid: OK`: the AI answered in the right format
- `[Gruppe 1]: German 2x, English 2x -> OK`: the placeholder was kept
- `Paragraphs: German 2, English 2`: the paragraph structure was kept
- `Length ratio: 101% -> OK`: nothing was left out

**❗ If a line says CHECK!**, look at the translation in STEP 5 and tell the group / Claude.

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("⚠️ Please run STEP 1 first (or click Run All at the top).")

de = sample.loc[0, "text"]; en = sample.loc[0, "translation"] or ""
print("JSON valid:", "OK" if sample.loc[0, "json_ok"] else "CHECK raw_output!")
n_de = len(re.findall(r"\[Gruppe 1\]", de))
n_en = len(re.findall(r"\[Gruppe 1\]", en))
ratio = len(en) / len(de)
print(f"[Gruppe 1]: German {n_de}x, English {n_en}x ->", "OK" if n_de == n_en else "CHECK!")
print(f"Paragraphs: German {de.count(chr(10)+chr(10))+1}, English {en.count(chr(10)+chr(10))+1}")
print(f"Length ratio: {ratio:.0%} ->", "OK" if 0.75 <= ratio <= 1.25 else "CHECK!")
print(f"Estimate for 5,000 paragraphs, one at a time: about {seconds*5000/3600:.1f} hours "
      f"(faster with parallel calls, as bacchuss does with multithreading)")

---
## STEP 7: Save the result

**👉 What to do:** click ▶ on the cell below.

**✅ You should see:** `Saved translation_test_result.xlsx and .json`

**Where to find it:** in the folder **Downloads → CAC_translation_test**, open **`translation_test_result.xlsx`**. It has the German text and the English translation side by side.

---
### 🎉 Done! Your task (testing the translation of one paragraph) is complete.

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("⚠️ Please run STEP 1 first (or click Run All at the top).")

sample.to_excel("translation_test_result.xlsx", index=False)
sample.to_json("translation_test_result.json", orient="records", force_ascii=False, indent=1)
print("Saved translation_test_result.xlsx and .json")